In [1]:
# turn off pretty printing, because the slides can't handle the verticality
%pprint

Pretty printing has been turned OFF


In [2]:
# load supriya's ipython extension to capture audio/graphs
%load_ext supriya.ext.ipython

In [3]:
# kill any running scsynth/supernova servers
from supriya.scsynth import kill

kill()

In [4]:
# turn soundcheck on
import supriya, time

server = supriya.Server().boot()
with server.at():
    with server.add_synthdefs(supriya.default):
        server.add_synth(supriya.default)

In [5]:
# turn soundcheck off
_ = server.quit()

March 18th, 2025

# Supriya: a Python API for SuperCollider

Joséphine Wolf Oberholtzer (she/her) <br/>
https://josephine-wolf-oberholtzer.com/ <br/>
https://github.com/supriya-project/supriya/

## Preamble

### Who am I

- ~16 years of experience in Python
- PhD from Harvard in music composition
    - massively multichannel tape music
    - symbolic composition in Python & LilyPond via Abjad
- Worked at Forced Exposure, Discogs, Capital One, Cortico/MIT

### What Supriya is for

- A foundational layer for application code
- Consistent API for realtime and non-realtime
- Consistent API for threaded and async concurrency models
- Reified server command API
- Musical-time-aware clocks
- Let's just make it easy to use SuperCollider inside the Python ecosystem!

### What Supriya isn't for

- Recreating sclang's class library
- Live coding (but you can try!)
- UIs (other libraries exist)
- IDEs (other libraries exist)
- MIDI (other libraries exist)
- etc.

### Why Python?

- Millions of people use it (visibility / support is good)
- Huge vibrant ecosystem (most things I want, somebody already invented and still support)
- (Relatively) simple syntax
- Good developer experience (tracebacks, debugging, testing, linting, type checking, CI, docs tooling, etc.)

### What I'm _not_ gonna cover

- How to use Python
- How to use SuperCollider
- How to install Supriya
   - We don't have time!
   - Ask me after!
   - https://supriya-project.github.io/supriya/installation.html
- Making really beautiful noises

### What I _am_ gonna cover

- Basic usage
- Design principles

### Design principles?

- Avoid globals, avoid implicitness
- Make it ~boring~ simple (avoid multiple means to the same ends, resist flexibility, "there should be one (and preferably only one) obvious way to do it")
- Make it verbose (give everything names, avoid abbreviations, prefer keywords over positionals, avoid fanciful names, strive for using terms of art aligned with the wider language ecosystem)
- Make it similar (strive for similar if not identical means of interacting with similar things)
- Make it narrowly focused (don't have to _implement_ what's already implemented in the rest of the ecosystem, but may have to _integrate_ with it)
- Make it easily introspectable
- Make it easily testable

My fervent hope is that everything you see here is utterly unsurprising.

## Realtime contexts: Servers

### Import supriya

In [ ]:
# supriya is a package, so let's import it
import supriya

In [ ]:
# everything in python is an object, so we can do some inspecting
supriya

In [ ]:
# let's look at the names defined inside the supriya namespace
dir(supriya)

### Import `Server`

In [ ]:
# we can import individual names out of supriya's namespace
from supriya import Server

In [ ]:
# instantiate the server
server = Server()

In [ ]:
# let's look at the names inside the Server namespace
dir(server)

In [ ]:
# actually let's look at the public name inside the Server namespace
[name for name in dir(server) if not name.startswith("_")]

In [ ]:
server

### Server options

In [ ]:
# TODO: Make Options a dataclass
server.options

In [ ]:
!scsynth -h

### Boot the server

In [ ]:
server.boot()

### Query the server

In [ ]:
server.status

In [ ]:
print(tree := server.query_tree())

In [ ]:
# this is actually a query tree object, not just a string
# ... which is helpful in more complex unit testing situations
# ... because the tree can be annotated with information beyond what scsynth provides
# ... but we can still use string comparisons
tree

### Quit the server

In [ ]:
server.quit()

### Boot with options

In [ ]:
# recall all the options from before
server.options

In [ ]:
# we can use those keywords to configure new options when booting, rebooting, quitting, etc.
server.boot(maximum_logins=2)

### Multiple users

In [ ]:
# We can create a handle to a second server proxy,
# pointed back at the same IP address and port as the first
other_server = Server()
other_server

In [ ]:
server  # note the port is the same as `other_server`

In [ ]:
other_server.connect()  # connect to the original server via .connect()

In [ ]:
print(f"{server.client_id=}")
print(f"{other_server.client_id=}")

In [ ]:
other_server.disconnect()  # disconnect from the original server

In [ ]:
server  # the original server remains online

### Lifecycle events

In [ ]:
for event_type in supriya.ServerLifecycleEvent:
    print(repr(event_type))

In [ ]:
def on_event(event):
    print(event)

for event_type in supriya.ServerLifecycleEvent:
    server.register_lifecycle_callback(event_type, on_event)
    other_server.register_lifecycle_callback(event_type, on_event)

In [ ]:
server.reboot()

In [ ]:
# this will panic!
other_server.boot()

In [ ]:
server.quit()

## Context Entities

... entities that live inside a synthesis _context_...

... Groups, Synths, Busses and Buffers ...

### Groups

In [ ]:
server = Server().boot()

In [ ]:
# add a group
group = server.add_group()
group

In [ ]:
print(dir(group))

In [ ]:
print(server.query_tree())

In [ ]:
# add a group to the group
child_group = group.add_group()
print(server.query_tree())

In [ ]:
# move the child group into the default group
child_group.move(target_node=server.default_group, add_action="ADD_TO_TAIL")
print(server.query_tree())

In [ ]:
# free the original parent group
group.free()
print(server.query_tree())

### Synths

In [ ]:
# add a synth (this will fail, silently)
synth = child_group.add_synth(synthdef=supriya.default)

In [ ]:
# we have a proxy to a synth
synth

In [ ]:
# but nothing on the server, because the request failed
print(server.query_tree())

#### Completions

In [ ]:
# using completion context manager
# executes inside a context moment (.at())
with server.at():
    with server.add_synthdefs(supriya.default):
        synth = child_group.add_synth(synthdef=supriya.default, frequency=666)

In [ ]:
server.sync()
print(server.query_tree())

In [ ]:
synth.free()

### Buses

In [ ]:
# add a control bus
control_bus = server.add_bus()
control_bus

In [ ]:
# add an audio bus
audio_bus = server.add_bus("audio")
audio_bus

In [ ]:
# add a bus group
control_bus_group = server.add_bus_group(count=4)
control_bus_group

In [ ]:
# a bus group aggregates together bus objects
for bus in control_bus_group:
    print(bus)

#### Shared Memory

In [ ]:
another_bus_group = server.add_bus_group(count=4)

In [ ]:
server.shared_memory

In [ ]:
server.shared_memory[another_bus_group]

In [ ]:
server.shared_memory[another_bus_group] = [0.1, 0.5, 0.3, 0.2]

In [ ]:
shared_memory._shm[another_bus_group]

### Buffers

In [ ]:
# add a mono buffer with 64 frames
buffer = server.add_buffer(channel_count=1, frame_count=64)

In [ ]:
# query the buffer
buffer.query()

In [ ]:
# generate a chebyshev polynomial in wavetable format
buffer.generate("cheby", amplitudes=[0.1, 0.2, 0.05], as_wavetable=True)

In [ ]:
# get values at indices in the buffer
buffer.get(0, 2, 4, 8)

In [ ]:
# get a range of values
buffer.get_range(index=0, count=16)

In [ ]:
# import plot rendering helper
from supriya import plot

In [ ]:
# plot the buffer
plot(buffer)

In [ ]:
# read a file into a buffer
file_path = supriya.assets_path / "audio/birds/birds-01.wav"
other_buffer = server.add_buffer(file_path=file_path)
plot(other_buffer)

In [ ]:
# import play helper
from supriya import play

In [ ]:
# normally play() isn't async
# but for fussy reasons related to jupyter itself being async,
# we have to await here
await play(other_buffer)

In [ ]:
# allocate a group of buffers, e.g.
buffer_group = server.add_buffer_group(count=4, frame_count=512, channel_count=1)
print(buffer_group)
for buffer_ in buffer_group:
    print(buffer_)

In [ ]:
# free all the buffers
buffer.free()
other_buffer.free()
buffer_group.free()

## Non-realtime contexts: Scores

In [ ]:
from supriya import Score

(score := Score())

In [ ]:
# inspect the score's namespace
# note: no queries, only mutations
[name for name in dir(score) if not name.startswith("_")]

In [ ]:
# add a synthdef at timestamp 0
with score.at(0):
    score.add_synthdefs(supriya.default)

# strum a series of synths
synths = []
for i in range(12):
    with score.at(i / 4):
        frequency = 111 * (i + 1)
        synth = score.add_synth(synthdef=supriya.default, frequency=frequency)
        synths.append(synth)

# free all of them
with score.at(4):
    for synth in synths:
        synth.free()

# pad out a no-op while the envelopes decay
with score.at(5):
    score.do_nothing()

In [ ]:
# play the score (and capture into the notebook)
from supriya import play

_ = await play(score)

In [ ]:
# render the score, returning the path and exit code
await score.render()

In [ ]:
# this will error!
# no queries, only mutations
with score.at(0):
    score.query_tree()

In [ ]:
# iterate the osc bundles in the score
for bundle in score.iterate_osc_bundles():
    print(repr(bundle))

In [ ]:
# but actually we store an intermediate format... requests
for bundle in score.iterate_request_bundles():
    print(bundle)

## OSC

### OSC messages & bundles

In [ ]:
from supriya import OscMessage, OscBundle

In [ ]:
message = OscMessage("/this", "is", "a", "message?", 1, 2.5, False)

In [ ]:
# print the interpretation representation
print(repr(message))

In [ ]:
# print the sclang-style hex representation
print(message)

In [ ]:
bundle = OscBundle(timestamp=666.23, contents=[message, message])

In [ ]:
# print the interpretation representation
print(repr(bundle))

In [ ]:
# print the sclang-style hex representation
print(bundle)

### Sending messages

In [ ]:
server.send(["/g_queryTree", 0])

### OSC Callbacks

In [ ]:
captured_messages = []

def procedure(message):
    captured_messages.append(message)

print(callback := server.register_osc_callback(procedure=procedure, pattern=["/n_end"]))

In [ ]:
server.send(["/n_free", 1008])

In [ ]:
captured_messages

In [ ]:
server.unregister_osc_callback(callback)

### Capturing IO

In [ ]:
# OSC is actually handled by a "protocol" class
server.osc_protocol

In [ ]:
with server.osc_protocol.capture() as transcript:
    with server.at() as moment:
        group = server.add_group()
        subgroup = group.add_group()
        with server.add_synthdefs(supriya.default) as completion:
            synth = subgroup.add_synth(synthdef=supriya.default, frequency=666)
    server.sync()

In [ ]:
for entry in transcript:
    print(entry)

In [ ]:
# the raw message is something else... an intermediate format
transcript[0].raw_message

In [ ]:
moment

In [ ]:
completion

## Synthdefs

### Building SynthDefs (1)

In [ ]:
r"""
SynthDef(\simple, { arg out=0, freq=440;
	Out.ar(out, SinOsc.ar(freq));
});
"""

In [ ]:
# A simple SynthDef using the builder pattern
from supriya.ugens import SynthDefBuilder
from supriya.ugens import Out, SinOsc

with SynthDefBuilder(freq=440, out=0) as builder:
    source = SinOsc.ar(frequency=builder["freq"])
    Out.ar(bus=builder["out"], source=source)

simple_synthdef = builder.build(name="simple")
simple_synthdef

### Graphing SynthDefs

In [ ]:
# a YAML-like textual representation
# this is useful for unit tests!
print(simple_synthdef)

In [ ]:
# a GraphViz representation
from supriya import graph

_ = graph(simple_synthdef)

### SynthDef internals

In [ ]:
dir(simple_synthdef)

In [ ]:
simple_synthdef.name

In [ ]:
simple_synthdef.anonymous_name

In [ ]:
simple_synthdef.effective_name

In [ ]:
simple_synthdef.parameters

In [ ]:
simple_synthdef.ugens

In [ ]:
simple_synthdef.has_gate

### Building SynthDefs (2)

In [ ]:
r"""
*makeDefaultSynthDef {
    SynthDef(\default, { arg out=0, freq=440, amp=0.1, pan=0, gate=1;
        var z;
        z = LPF.ar(
            Mix.new(VarSaw.ar(freq + [0, Rand(-0.4,0.0), Rand(0.0,0.4)], 0, 0.3, 0.3)),
            XLine.kr(Rand(4000,5000), Rand(2500,3200), 1)
        ) * Linen.kr(gate, 0.01, 0.7, 0.3, 2);
        OffsetOut.ar(out, Pan2.ar(z, pan, amp));
    }, [\ir]).add;
}
"""

In [ ]:
# More imports
from supriya.enums import DoneAction, ParameterRate
from supriya.ugens import (
    LPF,
    Linen,
    Mix,
    OffsetOut,
    Pan2,
    Parameter,
    Rand,
    SynthDefBuilder,
    VarSaw,
    XLine,
)

In [ ]:
# Define a builder
builder = SynthDefBuilder(
    out=Parameter(rate=ParameterRate.SCALAR, value=0),
    amplitude=0.1,
    frequency=440,
    gate=1,
    pan=0.5,
)

In [ ]:
# Use the builder as a context manager
with builder:
    linen = Linen.kr(
        attack_time=0.01,
        done_action=DoneAction.FREE_SYNTH,
        gate=builder["gate"],
        release_time=0.3,
        sustain_level=0.7,
    )

In [ ]:
# Use the builder again
with builder:
    low_pass = LPF.ar(
        source=Mix.new(
            VarSaw.ar(
                frequency=builder["frequency"]
                + (
                    0,
                    Rand.ir(minimum=-0.4, maximum=0.0),
                    Rand.ir(minimum=0.0, maximum=0.4),
                ),
                width=0.3,
            )
        )
        * 0.3,
        frequency=XLine.kr(
            start=Rand.ir(minimum=4000, maximum=5000),
            stop=Rand.ir(minimum=2500, maximum=3200),
        ),
    )

In [ ]:
# And again and again
with builder:
    panner = Pan2.ar(
        source=low_pass * linen * builder["amplitude"], position=builder["pan"]
    )

with builder:
    OffsetOut.ar(bus=builder["out"], source=panner)

In [ ]:
default = builder.build(name="default")
default

In [ ]:
_ = graph(default)

### The `synthdef` decorator

In [ ]:
# N.B. I'm not fond of this one because of
# a) how magical it is (not very, but just enough) but mainly because
# b) it makes type-checking difficult
from supriya.ugens import synthdef

In [ ]:
@synthdef("ir")
def default_decorated(out=0, amplitude=0.1, frequency=440, gate=1, pan=0.5):
    linen = Linen.kr(
        attack_time=0.01,
        done_action=DoneAction.FREE_SYNTH,
        gate=gate,
        release_time=0.3,
        sustain_level=0.7,
    )
    low_pass = LPF.ar(
        source=Mix.new(
            VarSaw.ar(
                frequency=frequency
                + (
                    0,
                    Rand.ir(minimum=-0.4, maximum=0.0),
                    Rand.ir(minimum=0.0, maximum=0.4),
                ),
                width=0.3,
            )
        )
        * 0.3,
        frequency=XLine.kr(
            start=Rand.ir(minimum=4000, maximum=5000),
            stop=Rand.ir(minimum=2500, maximum=3200),
        ),
    )
    panner = Pan2.ar(
        source=low_pass * linen * amplitude, position=pan
    )
    _ = OffsetOut.ar(bus=out, source=panner)

In [ ]:
default_decorated

In [ ]:
from supriya.ugens import Out, SinOsc

@synthdef()
def foo(out=0):
    print(repr(out))
    _ = Out.ar(source=SinOsc.kr())

foo

### UGen methods

In [ ]:
dir(SinOsc)

### SynthDef (de)compilation

In [ ]:
# SynthDefs compile to byte strings
compiled = default.compile()
compiled

In [ ]:
# valid byte strings can be decompiled back into SynthDefs
from supriya.ugens import decompile_synthdef

decompiled = decompile_synthdef(compiled)
decompiled

In [ ]:
# sanity-check: the decompiled SynthDef is not the same in memory
default is decompiled

### Compiling via sclang

In [ ]:
# Supriya provides utilities for compiling via sclang.
# This is intended for validating its own logic vs sclang (as a reference spec).
from supriya.ugens import SuperColliderSynthDef

sc_synthdef = SuperColliderSynthDef(
    "foo", "Out.ar(0, SinOsc.ar(freq: 420) * SinOsc.ar(freq: 440))"
)
sc_compiled_synthdef = sc_synthdef.compile()  # return bytes
sc_compiled_synthdef

In [ ]:
# The sclang-derived SynthDef byte string can be decompiled back into a SynthDef.
print(decompile_synthdef(sc_compiled_synthdef))

### UGen metaprogramming (MAYBE CUT)

In [ ]:
from supriya.ugens import UGen, param, ugen

# A dupe of SinOsc
@ugen(ar=True, kr=True, is_pure=True)
class AnotherSinOsc(UGen):
    frequency = param(440.0)
    phase = param(0.0)

In [ ]:
AnotherSinOsc.ar()

In [ ]:
AnotherSinOsc.kr()

In [ ]:
# This won't work because ir=True wasn't set
AnotherSinOsc.ir()

In [ ]:
# A dupe of Out
@ugen(ar=True, kr=True, is_output=True, channel_count=0, fixed_channel_count=True)
class AnotherOut(UGen):
    bus = param(0)
    source = param(unexpanded=True)

AnotherOut.ar(source=AnotherSinOsc.ar())

In [ ]:
from supriya.ugens.pv import PV_ChainUGen

# A dupe of PV_BinShift
@ugen(kr=True, is_width_first=True)
class AnotherPV_BinShift(PV_ChainUGen):
    pv_chain = param()
    stretch = param(1.0)
    shift = param(0.0)
    interpolate = param(0)

# This won't work because of missing pv_chain argument
AnotherPV_BinShift.kr()

In [ ]:
"""
def ugen(
    *,
    ar: bool = False,
    kr: bool = False,
    ir: bool = False,
    dr: bool = False,
    new: bool = False,
    has_done_flag: bool = False,
    is_input: bool = False,
    is_multichannel: bool = False,
    is_output: bool = False,
    is_pure: bool = False,
    is_width_first: bool = False,
    channel_count: int = 1,
    fixed_channel_count: bool = False,
    signal_range: Optional[int] = None,
) -> Callable[[Type["UGen"]], Type["UGen"]]:
    ...
"""

## Clocks

In [ ]:
from supriya import Clock

In [ ]:
(clock := Clock())

In [ ]:
clock.beats_per_minute

In [ ]:
clock.time_signature

In [ ]:
def clock_callback(context):
    print(context)
    if context.event.invocations == 4:
        return None
    return 0.25

In [ ]:
clock.schedule(clock_callback)
clock.start()

In [ ]:
clock.stop()

## Patterns

In [ ]:
from supriya.patterns import SequencePattern, RandomPattern, ShufflePattern

In [ ]:
sequence = SequencePattern([111, 150, 180])

In [ ]:
for x in sequence: print(x)

In [ ]:
from supriya.patterns import EventPattern

In [ ]:
from supriya.patterns import FxPattern, ParallelPattern

In [ ]:
# let's make some pretty SynthDefs (finally!)
# two event patterns with fx tails (different ones) played in parallel
# with a final fx reverb tail
# i want envelopes for lp filters
# i want delays (weird ones!)
# i want ghostly frequency shifting

## Async Concurrency

from https://docs.python.org/3/library/asyncio.html:

> asyncio is a library to write concurrent code using the async/await syntax.

> asyncio is used as a foundation for multiple Python asynchronous frameworks that provide high-performance network and web-servers, database connection libraries, distributed task queues, etc.

> asyncio is often a perfect fit for IO-bound and high-level structured network code.

In [ ]:
from supriya import AsyncServer, find_free_port

async_server = AsyncServer()

In [ ]:
(coro := async_server.boot(port=find_free_port()))

In [ ]:
await coro

In [ ]:
with async_server.at():
    group = async_server.add_group()
    with async_server.add_synthdefs(supriya.default):
        synth = group.add_synth(synthdef=supriya.default, frequency=123)

In [ ]:
print(await async_server.query_tree())

In [ ]:
await async_server.quit()

In [ ]:
from supriya import AsyncClock, AsyncOfflineClock, OfflineClock

## Contexts

- Mutation interface (all contexts)
- Query interface (realtime only)
- Mutations are realtime/nonrealtime agnostic
- Mutations are concurrency agnostic (".send()" is _always_ sync)
- We can write code targeted against "Context" regardless of what kind

## Cleanliness

- docs
- typing
- testing
- ci/cd

## Future work?

- More docs
- More examples
- DAW affordances
    - multi-context mixers
    - tracks & subtracks
    - send & receives
    - modulation
    - transport

## Ciao ciao!

Questions?

Thanks, darlings! <br/>
xoxo, joséphine

https://josephine-wolf-oberholtzer.com/ <br/>
https://github.com/supriya-project/supriya/